# 📖 第八课：KNN 与朴素贝叶斯

**目标**：掌握两种经典分类算法——KNN（基于距离的懒惰学习）和朴素贝叶斯（基于概率的快速分类器）

---

## 📚 学习目标
- 理解 KNN 的核心思想：「近朱者赤」——让邻居投票决定分类
- 掌握 K 值选择、距离度量对结果的影响
- 理解朴素贝叶斯的概率框架：用贝叶斯定理做分类
- 理解「朴素」的含义——条件独立假设
- 对比两种算法的适用场景与优缺点

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.font_manager as fm
import pathlib
_cache = matplotlib.get_cachedir()
for _f in pathlib.Path(_cache).glob('fontlist*.json'):
    _f.unlink(missing_ok=True)
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.sans-serif'] = ['PingFang HK', 'PingFang SC', 'Heiti TC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report



## 🔍 KNN：K-最近邻分类器

### 核心直觉

KNN 的哲学极其简单：**物以类聚**。

给定一个新样本，看它最近的 K 个邻居是什么类别，少数服从多数。

```
新样本 → 计算到所有训练样本的距离 → 取最近的 K 个 → 投票决定类别
```

### 为什么叫「懒惰学习」？

KNN 没有训练过程！它把所有训练数据原封不动存着，等到预测时才计算。
对比：线性回归是「急切学习」——训练时就学出参数，预测时只做简单计算。

### 关键参数
- **K 值**：太小（如 K=1）→ 过拟合；太大（如 K=N）→ 欠拟合。经验值：√n
- **距离度量**：欧氏距离（默认）、曼哈顿距离、闵可夫斯基距离

In [ ]:
# KNN 实战：鸢尾花分类 + 不同 K 值对比
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 测试不同 K 值
k_values = range(1, 21)
train_acc = []
test_acc = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc.append(knn.score(X_train, y_train))
    test_acc.append(knn.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(k_values, train_acc, 'o-', label='训练准确率')
plt.plot(k_values, test_acc, 's-', label='测试准确率')
plt.xlabel('K 值')
plt.ylabel('准确率')
plt.title('KNN：K 值对准确率的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(test_acc)]
print(f'最佳 K 值: {best_k}, 测试准确率: {max(test_acc):.3f}')

## 🎲 朴素贝叶斯分类器

### 核心思想：用贝叶斯定理做分类

贝叶斯定理：

```
P(类别|特征) = P(特征|类别) × P(类别) / P(特征)
```

**「朴素」在哪？** 假设所有特征之间相互独立。

这意味着：
```
P(特征|类别) = P(特征1|类别) × P(特征2|类别) × ... × P(特征n|类别)
```

这个假设几乎永远不成立，但朴素贝叶斯在实践中表现往往出奇地好！

### 为什么「朴素」假设还能work？
因为分类只关心哪个类别的概率最大，不关心概率的精确值。
只要排序对，即使绝对概率有偏差也没关系。

### 三种常见变体
- **GaussianNB**：特征是连续值（假设高斯分布）
- **MultinomialNB**：特征是计数（如词频）
- **BernoulliNB**：特征是二值（如词是否出现）

In [ ]:
# 朴素贝叶斯实战 + 与 KNN 对比
# 使用鸢尾花数据集

# Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)
nb_pred = gnb.predict(X_test)

# KNN (最佳K)
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
knn_pred = knn_best.predict(X_test)

print('=== 朴素贝叶斯 ===')
print(f'准确率: {accuracy_score(y_test, nb_pred):.3f}')
print(classification_report(y_test, nb_pred, target_names=['setosa', 'versicolor', 'virginica']))

print('=== KNN (K={}) ==='.format(best_k))
print(f'准确率: {accuracy_score(y_test, knn_pred):.3f}')
print(classification_report(y_test, knn_pred, target_names=['setosa', 'versicolor', 'virginica']))

In [ ]:
# 交叉验证对比两种方法的稳定性
cv_knn = cross_val_score(KNeighborsClassifier(n_neighbors=best_k), X, y, cv=5)
cv_nb = cross_val_score(GaussianNB(), X, y, cv=5)

print('5折交叉验证结果：')
print(f'KNN:  {cv_knn.mean():.3f} ± {cv_knn.std():.3f}')
print(f'NB:   {cv_nb.mean():.3f} ± {cv_nb.std():.3f}')

# 特征独立性检验（直观展示「朴素」假设的近似程度）
import pandas as pd
df = pd.DataFrame(X, columns=['花萼长','花萼宽','花瓣长','花瓣宽'])
df['类别'] = y
print('\n特征相关性矩阵：')
print(df.iloc[:,:4].corr().round(2))

## 📊 两种算法对比总结

| 维度 | KNN | 朴素贝叶斯 |
|------|-----|------------|
| 学习方式 | 懒惰学习（无训练） | 急切学习（有训练） |
| 预测速度 | 慢（需计算所有距离） | 快（只需查概率表） |
| 核心假设 | 相似样本属于同类 | 特征条件独立 |
| 适用场景 | 低维、特征量少 | 高维、文本分类 |
| 抗噪能力 | K大时较鲁棒 | 对无关特征鲁棒 |
| 代表应用 | 推荐系统、图像检索 | 垃圾邮件检测 |

### AI 演进中的位置

这两种算法都是 1950-60 年代的产物，代表了早期 AI 的两条路径：
- **KNN** → 基于实例的推理（影响了后续的案例推理、局部学习）
- **朴素贝叶斯** → 基于概率的推理（影响了贝叶斯网络、概率图模型）

朴素贝叶斯在 NLP 中的地位尤其重要——在深度学习出现前，它是文本分类的标配算法。

---

**下节课预告**：聚类与 KMeans——当没有标签时，如何让数据自己分组？